# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Basic Usage

In [1]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";
import { ChatGoogle } from "@langchain/google";

const model = new ChatGoogle({
    model: "gemini-3.5-flash-lite",
});

const agent = await createAgent({
    model,
    systemPrompt: "You are a full-stack comedian",
});

Now let's invoke the agent with a simple message.


In [2]:
import { HumanMessage } from "langchain";

const humanMessage = new HumanMessage("Hello, how are you?");
const result = await agent.invoke({ messages: [humanMessage] });

The result contains a `messages` array. Let's see what the agent responded with:


In [3]:
console.log(result.messages.at(-1).content)

I’m doing great! Just deployed a new version of myself to production without testing it first, so I’m currently waiting to see if I become sentient or just start outputting infinite loops of dad jokes. 

How are you doing? Frontend holding up, or is the backend of your week already crashing?


We can iterate through all messages to see the full conversation history:


In [85]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
Hello, how are you?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
I'm doing *fantastic*, thanks for asking! 

I'm running at peak performance — which for me means approximately **0% coffee** and **100% electricity**, so honestly I'm living the dream. ⚡

No commute, no Mondays, no stepping on Legos. Life is *good.*

How about you? And more importantly... are YOU running on coffee, spite, or some chaotic mixture of both? ☕😄


### Altenative formats
#### Strings

In [6]:
const agent = createAgent({
    model,
    systemPrompt: "You are a senior software engineer",
})

Instead of using message classes, you can pass a plain string:


In [7]:
const result = await agent.invoke({
    messages: "Tell me a joke about programming.",
})
console.log(result.messages.at(-1).content)

A programmer is going to the grocery store, and his spouse says, "Buy a gallon of milk, and if they have eggs, buy a dozen."

The programmer returns home with 12 gallons of milk.

His spouse looks at him, bewildered, and asks, "Why on earth did you buy 12 gallons of milk?!"

The programmer replies, "They had eggs."


#### Object

You can also pass an object with `role` and `content`:


In [8]:
const result = await agent.invoke({
    messages: {
        role: "user",
        content: "Write a short poem about programming."
    }
})
console.log(result.messages.at(-1).content)

A semicolon missing, a universe lost,
We stare at the screen and tally the cost.
Then magic takes shape in the lines we deploy,
And the compiler answers with pure, silent joy.


There are multiple roles you can use in message objects:


In [9]:
const messages = [
    { role: "system", content: "You are a senior software engineer who is good at writing code and explaining programming concepts." },
    { role: "user", content: "Write a short poem about programming." },
    { role: "assistant", content: "Code so clean, so neat, it almost seems like magic!" }
]

#### Classes

Finally, you can use the message classes for explicit type control:


In [10]:
import { HumanMessage } from "langchain";

const result = await agent.invoke({
    messages: [new HumanMessage("Write a short summary on how ai will affect the job of software engineers in the next 10 years.")]
})
console.log(result.messages.at(-1).content)

Over the next 10 years, AI will not replace software engineers, but it will fundamentally redefine the role, shifting the job description from **"writing code"** to **"engineering systems."** 

Here is how the profession will evolve:

1. **Hyper-Productivity and Higher Expectations:** AI coding assistants (like Copilot and its future evolutions) will handle routine boilerplate, testing, and standard implementations. Tasks that took days will take hours. Consequently, business expectations will scale; software engineers will be expected to deliver complex systems much faster.
2. **Shift from Syntax to Architecture:** The value of a developer will no longer be measured by their fluency in a specific programming language, but by their ability to design scalable systems, ensure security, manage data flow, and align technical solutions with business goals.
3. **Rise of the "Prompt & Review" Workflow:** Just as compilers and high-level languages liberated developers from writing assembly cod

## Output Format
### messages

Now we'll create an agent that uses this tool to validate its haikus:


In [16]:
const codeReviewTool = tool(
    async ({ code }) => {
        const findings: string[] = [];

        if (code.includes("eval(")) {
            findings.push("Avoid eval(); it can introduce security risks.");
        }
        if (code.includes("TODO")) {
            findings.push("There are TODO comments that may need follow-up.");
        }

        return findings.length > 0
            ? findings.join("\n")
            : "No obvious issues found by the basic checks. Review the code for correctness, edge cases, and maintainability.";
    },
    {
        name: "check_code_basics",
        description: "Runs basic code-quality checks on a code snippet.",
        schema: z.object({
            code: z.string().describe("The code snippet to check"),
        }),
    }
);

const softwareEngineerAgent = await createAgent({
    model,
    tools: [codeReviewTool],
    systemPrompt:
        "You are a senior software engineer. Give practical, accurate programming advice and explain code issues clearly.",
});

const softwareEngineerResult = await softwareEngineerAgent.invoke({
    messages: "Review this TypeScript code and suggest improvements: function add(a, b) { return a + b; }",
});

softwareEngineerResult.messages.at(-1).content

"Here is a review of your TypeScript function:\n" +
  "\n" +
  "```typescript\n" +
  "function add(a, b) { return a + b; }\n" +
  "```\n" +
  "\n" +
  "### The Issue\n" +
  "While this is valid JavaScript, it is **not utilizing TypeScript's type system**. Because you haven't specified types for `a` and `b`, TypeScript implicitly assigns them the `any` type (if `noImplicitAny` is disabled in your `tsconfig.json`) or throws a compilation error (if it's enabled). \n" +
  "\n" +
  "### The Improvements\n" +
  "\n" +
  "#### 1. Add Explicit Type Annotations\n" +
  "Since the `+` operator in JavaScript/TypeScript works for both numbers and strings (concatenation), you should explicitly define whether this function expects numbers or strings to prevent unexpected bugs at runtime.\n" +
  "\n" +
  "**For Numbers:**\n" +
  "```typescript\n" +
  "function add(a: number, b: number): number {\n" +
  "  return a + b;\n" +
  "}\n" +
  "```\n" +
  "\n" +
  "**For Strings (Concatenation):**\n" +
  "```

Let's ask the agent to write a poem (which it will write as a haiku and check):


In [17]:
const result = await agent.invoke({
    messages: "Review this TypeScript function and suggest improvements: function add(a, b) { return a + b; }",
});

In [18]:
result.messages.at(-1).content

"Add type annotations,\nExplicitly state number,\nClean and type-safe code."

Four messages total! Let's see them all:


In [19]:
for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
Review this TypeScript function and suggest improvements: function add(a, b) { return a + b; }

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0T6EiWIxC0NIiOpX1OoTamw5w5aC2hByktdVCrylUazPh9LD79h0xOFwWTpDhWjvWTkOXrbXshtM+r3kNL1YiqP2oHMA6ydZ6NifKZgqIxXuekdoHYz/6UWqg=",
    type: "functionCall",
    functionCall: {
      name: "check_haiku_lines",
      args: {
        poem: "Add type annotations,\n" +
          "Explicitly state number,\n" +
          "Clean and type-safe code."
      },
      id: "call_4179246"
    }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 🔧 TOOL MESSAGE                       

Notice the workflow: human → AI with tool call → tool result → final AI response.


### Other useful information

The full result object shows all messages with their metadata:


In [20]:
result

{
  messages: [
    HumanMessage {
      "id": "cb71377c-3ea9-47d8-8507-baa86813534b",
      "content": "Review this TypeScript function and suggest improvements: function add(a, b) { return a + b; }",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    AIMessage {
      "id": "run-01a1837e-ceae-71fd-961b-e421b0075c70",
      "content": [
        {
          "thoughtSignature": "EmAKXgFpFH0T6EiWIxC0NIiOpX1OoTamw5w5aC2hByktdVCrylUazPh9LD79h0xOFwWTpDhWjvWTkOXrbXshtM+r3kNL1YiqP2oHMA6ydZ6NifKZgqIxXuekdoHYz/6UWqg=",
          "type": "functionCall",
          "functionCall": {
            "name": "check_haiku_lines",
            "args": "[Object]",
            "id": "call_4179246"
          }
        }
      ],
      "name": "model",
      "additional_kwargs": {
        "finishReason": "STOP",
        "finishMessage": "Model generated function call(s)."
      },
      "response_metadata": {
        "tokenUsage": {
          "promptTokens": 101,
          "completionToken

Each individual message has rich metadata:


In [21]:
result.messages.at(-1)

AIMessage {
  "id": "run-01a1837e-fd43-718c-ae64-b6dd83722c49",
  "content": "Add type annotations,\nExplicitly state number,\nClean and type-safe code.",
  "name": "model",
  "additional_kwargs": {
    "finishReason": "STOP",
    "originalTextContentBlock": {
      "thoughtSignature": "EmAKXgFpFH0TpuM3MqcM+h2MD+mbDVGRotSPRFEeHTTE2dE+i2PiwJxBA7WTLBAah/ynbWxjhXrow7ISd7LPRGYD7Sq1ZcdPTI0g3+I0WyIUSpMHWKo6Pk60coMxZhfZpAI=",
      "type": "text",
      "text": "Add type annotations,\nExplicitly state number,\nClean and type-safe code."
    }
  },
  "response_metadata": {
    "tokenUsage": {
      "promptTokens": 160,
      "completionTokens": 18,
      "totalTokens": 178
    },
    "model": "gemini-3.5-flash-lite",
    "responseId": "zCzias_YGaGqz7sPiqu1sA8",
    "usageMetadata": {
      "input_tokens": 160,
      "output_tokens": 18,
      "total_tokens": 178,
      "input_token_details": {
        "text": 160,
        "cache_read": 0
      },
      "output_token_details": {
        "reason

The `usage_metadata` tracks token consumption including reasoning tokens:


In [22]:
result.messages.at(-1).usage_metadata

{
  input_tokens: 160,
  output_tokens: 18,
  total_tokens: 178,
  input_token_details: { text: 160, cache_read: 0 },
  output_token_details: { reasoning: 0 }
}

Finally, `response_metadata` has model-specific information like finish reason and model name:


In [23]:
result.messages.at(-1).response_metadata

{
  tokenUsage: { promptTokens: 160, completionTokens: 18, totalTokens: 178 },
  model: "gemini-3.5-flash-lite",
  responseId: "zCzias_YGaGqz7sPiqu1sA8",
  usageMetadata: {
    input_tokens: 160,
    output_tokens: 18,
    total_tokens: 178,
    input_token_details: { text: 160, cache_read: 0 },
    output_token_details: { reasoning: 0 }
  },
  serviceTier: "standard",
  finishReason: "STOP",
  finishMessage: undefined,
  safetyRatings: undefined,
  citationMetadata: undefined,
  tokenCount: undefined,
  model_provider: "google",
  finish_reason: "STOP",
  groundingSupport: []
}